# ACNE04 Week-1 audit — complete

Extract if needed, inspect, audit. Standard library only. No GPU, no detector, no training.

| | |
|---|---|
| **SETUP** | tree, file types, image inventory, annotation preview, box format |
| **N1** | agreement between `rule(count)` and the released grade, published Hayashi divisions |
| **N2** | the cut-points that best reproduce the released grades, with a degeneracy check |
| **N3** | label-file count vs number of boxes |
| **N4** | subject-group structure and the leakage a random split causes |

Hayashi, Akamatsu & Kawashima, *J Dermatol* 2008;35(5):255–260 give the divisions as 0–5 mild, 6–20 moderate, 21–50 severe, >50 very severe, counting **inflammatory** eruptions on **half** the face. Those reached 75.1% conformity against a three-dermatologist panel — the reference point for what N1 should look like if ACNE04's grades were clinician-assigned.

**Read the INFERENCE REPORT** before trusting anything downstream. Verify the ACNE04 licence before use.

## 1. Config

In [1]:
from __future__ import annotations

import json
import random
import re
import tarfile
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

# ===========================================================================
# CONFIG
# ===========================================================================
ROOT = "/kaggle/input/datasets/atkiyaanisa/acne04/acne04"
OUT = "audit_out"
SEEDS = 20

# Only used if ROOT holds unextracted tarballs. On Kaggle the archives are
# already unpacked, so these are ignored.
CLASSIFICATION_TAR = None
DETECTION_TAR = None

# Optional ACNE04-v2 (AcneAI re-annotation) root. Separates annotation
# incompleteness from the half-face counting convention. None to skip.
V2_DETECTION_ROOT = None
# ===========================================================================

## 2. Helper functions

Nothing to edit. Run and move on.

In [2]:
HAYASHI_CUTS = (5, 20, 50)
N_GRADES = 4
CANONICAL = 1457
REPORTED_ZIP = 1513
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
ARCHIVE_EXTS = {".tar", ".gz", ".tgz", ".zip", ".bz2", ".xz", ".7z"}


def rule(count, cuts=HAYASHI_CUTS):
    """Map a lesion count to an ordinal grade 0..len(cuts), upper-inclusive."""
    g = 0
    for c in cuts:
        if count > c:
            g += 1
        else:
            break
    return g


# ---------------------------------------------------------------------------
# extraction
# ---------------------------------------------------------------------------

def safe_extract(tar_path, dest):
    """Extract a tarball, refusing members that would escape the destination."""
    tar_path = Path(tar_path)
    if not tar_path.exists():
        print(f"  MISSING: {tar_path}")
        return 0
    dest.mkdir(parents=True, exist_ok=True)
    resolved = dest.resolve()
    with tarfile.open(tar_path) as tf:
        members = tf.getmembers()
        for m in members:
            if not str((resolved / m.name).resolve()).startswith(str(resolved)):
                raise RuntimeError(f"unsafe path in archive: {m.name}")
        try:
            tf.extractall(dest, filter="data")   # Python 3.12+
        except TypeError:
            tf.extractall(dest)                   # paths checked above
    print(f"  extracted {len(members)} members from {tar_path.name}")
    return len(members)


# ---------------------------------------------------------------------------
# parsing
# ---------------------------------------------------------------------------

def find_annotation_files(root):
    return sorted(p for p in root.rglob("*")
                  if p.suffix.lower() in {".txt", ".csv"}
                  and p.is_file() and p.stat().st_size > 0)


def parse_annotation_file(path):
    """Return [(image_name, [ints on the line])] for lines that name an image."""
    rows = []
    try:
        text = path.read_text(errors="replace")
    except Exception:
        return rows
    for line in text.splitlines():
        parts = line.replace(",", " ").split()
        if not parts:
            continue
        name = next((t for t in parts if Path(t).suffix.lower() in IMAGE_EXTS), None)
        if name is None:
            continue
        ints = []
        for tok in parts:
            if tok == name:
                continue
            try:
                ints.append(int(float(tok)))
            except ValueError:
                pass
        rows.append((name, ints))
    return rows


def infer_columns(rows):
    """
    Decide which integer column is the grade and which is the lesion count.

    Heuristic, stated so it can be audited:
      - the grade column takes few distinct values, all within 0..N_GRADES-1
      - the count column has a wider range and a higher max
    """
    widths = Counter(len(i) for _, i in rows)
    width = widths.most_common(1)[0][0]
    cols = [[] for _ in range(width)]
    for _, ints in rows:
        if len(ints) == width:
            for i, v in enumerate(ints):
                cols[i].append(v)

    stats = []
    for i, c in enumerate(cols):
        stats.append(None if not c else
                     {"index": i, "min": min(c), "max": max(c),
                      "n_distinct": len(set(c)), "mean": sum(c) / len(c)})

    grade_idx = None
    for s in stats:
        if s and s["min"] >= 0 and s["max"] <= N_GRADES - 1 and s["n_distinct"] <= N_GRADES:
            grade_idx = s["index"]
            break

    count_idx, best = None, -1
    for s in stats:
        if s and s["index"] != grade_idx and s["max"] > best:
            best, count_idx = s["max"], s["index"]

    return grade_idx, count_idx, {"line_widths": dict(widths), "column_stats": stats}


def count_boxes(root):
    """Boxes per image, from VOC XML or from plain-text box files."""
    per_image = {}
    for p in root.rglob("*.xml"):
        try:
            t = ET.parse(p)
            fn = t.findtext("filename") or (p.stem + ".jpg")
            per_image[Path(fn).name] = len(t.findall(".//object"))
        except Exception:
            continue
    if per_image:
        return per_image
    tally = Counter()
    for p in find_annotation_files(root):
        for name, ints in parse_annotation_file(p):
            if len(ints) >= 4:
                tally[Path(name).name] += 1
    return dict(tally)


# ---------------------------------------------------------------------------
# N2
# ---------------------------------------------------------------------------

def fit_cuts(counts, grades, max_count):
    """
    Exhaustive search over 3 cut-points maximising exact agreement, via prefix
    sums so each candidate triple costs O(1):

        agreement(c1,c2,c3) = P0[c1] + (P1[c2]-P1[c1])
                            + (P2[c3]-P2[c2]) + (P3[M]-P3[c3])
    """
    M = max(0, min(max_count, 200))
    hist = [[0] * (M + 2) for _ in range(N_GRADES)]
    n_used = 0
    for n, g in zip(counts, grades):
        if 0 <= g < N_GRADES and 0 <= n <= M:
            hist[g][n] += 1
            n_used += 1
    if n_used == 0:
        return HAYASHI_CUTS, 0.0

    P = [[0] * (M + 2) for _ in range(N_GRADES)]
    for g in range(N_GRADES):
        run = 0
        for n in range(M + 1):
            run += hist[g][n]
            P[g][n] = run
        P[g][M + 1] = run

    total3 = P[3][M]
    best, best_hits = HAYASHI_CUTS, -1
    for c1, c2, c3 in combinations(range(M + 1), 3):
        hits = P[0][c1] + (P[1][c2] - P[1][c1]) + (P[2][c3] - P[2][c2]) + (total3 - P[3][c3])
        if hits > best_hits:
            best_hits, best = hits, (c1, c2, c3)
    return best, best_hits / n_used


def describe_cuts(counts, grades, cuts, label=""):
    """
    Report bin occupancy and flag a degenerate fit.

    The fit will collapse two cut-points together when a grade is rare, because
    an empty interval costs the objective nothing. Always check occupancy
    before quoting fitted cut-points.
    """
    occ = Counter(rule(n, cuts) for n in counts)
    rel = Counter(grades)
    print(f"{label}cuts {cuts}")
    print(f"{label}  bin occupancy (rule)     : {dict(sorted(occ.items()))}")
    print(f"{label}  released grade occupancy : {dict(sorted(rel.items()))}")
    warn = []
    widths = [cuts[i] - cuts[i - 1] for i in range(1, len(cuts))]
    if any(w <= 1 for w in widths):
        warn.append("adjacent cut-points: an interval is empty or near-empty")
    for g in range(N_GRADES):
        if rel.get(g, 0) < 30:
            warn.append(f"released grade {g} has only {rel.get(g, 0)} images - "
                        "its boundary is weakly identified")
    for w in warn:
        print(f"{label}  WARNING: {w}")
    if not warn:
        print(f"{label}  no degeneracy flags")
    return {"bin_occupancy": dict(occ), "warnings": warn}


# ---------------------------------------------------------------------------
# N4
# ---------------------------------------------------------------------------

STEM_RE = re.compile(r"^(?P<prefix>.*?)(?P<num>\d+)$")


def subject_groups(names, run_gap=1):
    """
    Group images whose numeric filename suffixes run consecutively within a
    prefix. Wen et al. (2022) report ACNE04 contains triplets with consecutive
    filenames, implying several photographs per subject.

    HEURISTIC, and it over-merges, because ACNE04 filenames carry the grade in
    the prefix and run consecutively within a grade. Upper bound on subjects,
    lower bound on leakage. Corroborate with an embedding near-duplicate check
    before publishing a number from it.
    """
    parsed = []
    for n in names:
        stem = Path(n).stem
        m = STEM_RE.match(stem)
        parsed.append((m.group("prefix"), int(m.group("num")), n) if m else (stem, -1, n))

    by_prefix = defaultdict(list)
    for prefix, num, n in parsed:
        by_prefix[prefix].append((num, n))

    groups, gid = {}, 0
    for _, items in by_prefix.items():
        items.sort()
        prev = None
        for num, n in items:
            if prev is None or num < 0 or (num - prev) > run_gap:
                gid += 1
            groups[n] = gid
            prev = num
    return groups


def leak_rate(groups, seed=0, test_frac=0.2):
    names = sorted(groups)
    random.Random(seed).shuffle(names)
    k = int(len(names) * test_frac)
    test, train = set(names[:k]), set(names[k:])
    train_groups = {groups[n] for n in train}
    leaked = sum(1 for n in test if groups[n] in train_groups)
    return leaked / max(1, len(test))

## 3. Run

One cell, all sections in order. Takes a minute or two — the cut-point search is exhaustive.

In [3]:
def main():
    root = Path(ROOT)
    out = Path(OUT)
    out.mkdir(parents=True, exist_ok=True)

    print("=" * 74)
    print("SETUP")
    print("=" * 74)
    existing = ([p for p in root.rglob("*")
                 if p.is_file() and p.suffix.lower() not in ARCHIVE_EXTS]
                if root.exists() else [])
    if existing:
        print(f"  {len(existing)} extracted files already present - skipping extraction")
    else:
        if CLASSIFICATION_TAR:
            safe_extract(CLASSIFICATION_TAR, root)
        if DETECTION_TAR:
            safe_extract(DETECTION_TAR, root)

    files = [p for p in root.rglob("*") if p.is_file()]
    assert files, f"nothing under {root} - check ROOT"

    print("\n  TREE (depth 2, recursive file counts)")
    for p in sorted(root.rglob("*")):
        if not p.is_dir():
            continue
        rel = p.relative_to(root)
        if len(rel.parts) > 2:
            continue
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        print(f"    {'    ' * (len(rel.parts) - 1)}{rel.as_posix()}/   ({n} files)")

    print("\n  FILE TYPES")
    for ext, n in Counter(p.suffix.lower() for p in files).most_common(12):
        print(f"    {ext or '(none)':<10} {n}")

    # ---- inventory -------------------------------------------------------
    print("\n" + "=" * 74)
    print("IMAGE INVENTORY")
    print("=" * 74)
    images = [p for p in files if p.suffix.lower() in IMAGE_EXTS]
    unique = {p.name for p in images}
    print(f"  image files on disk    : {len(images)}")
    print(f"  UNIQUE filenames       : {len(unique)}   <- the real inventory")
    print(f"  published canonical    : {CANONICAL}")
    print(f"  reported zip contents  : {REPORTED_ZIP}")

    by_dir = Counter(p.parent.relative_to(root).as_posix() for p in images)
    print("\n  images per directory:")
    for d, n in by_dir.most_common(12):
        print(f"    {n:>6}  {d}/")

    if len(images) != len(unique):
        print(f"\n  {len(images) - len(unique)} filename repeats across directories.")
        print("  Repeats between Classification/ and Detection/ are expected - both")
        print("  tars ship the same photographs. Only repeats WITHIN one branch, or")
        print("  an extra resized folder, indicate duplicate copies.")

    if len(unique) > REPORTED_ZIP:
        print(f"\n  *** {len(unique) - CANONICAL} MORE UNIQUE FILES THAN CANONICAL ***")
        print("  Likely the auxiliary 1024-resolution subsets: resized copies of the")
        print("  same photographs. A random split puts an image in train and its own")
        print("  resized copy in test. Exclude them before computing anything.")
    elif len(unique) == CANONICAL:
        print(f"\n  Unique count matches the canonical {CANONICAL}.")
    else:
        print(f"\n  Unique count is {len(unique)}, between the canonical {CANONICAL} and")
        print(f"  the reported {REPORTED_ZIP}. Part of the version question to reconcile.")

    suspicious = [d for d in by_dir if any(
        t in d.lower() for t in ("1024", "resize", "resized", "aux", "small", "512"))]
    if suspicious:
        print("\n  Directories whose names suggest resized copies:")
        for d in suspicious:
            print(f"    {d}/")

    # ---- annotations -----------------------------------------------------
    print("\n" + "=" * 74)
    print("ANNOTATION FILES")
    print("=" * 74)
    ann_files = find_annotation_files(root)
    all_rows, per_file = [], {}
    for p in ann_files:
        rows = parse_annotation_file(p)
        if rows:
            per_file[p.relative_to(root).as_posix()] = len(rows)
            all_rows.extend(rows)

    for p in ann_files[:8]:
        try:
            lines = p.read_text(errors="replace").splitlines()
        except Exception as e:
            print(f"\n  {p.relative_to(root).as_posix()}  (unreadable: {e})")
            continue
        print(f"\n  {p.relative_to(root).as_posix()}   ({len(lines)} lines)")
        for ln in lines[:5]:
            print(f"      {ln}")
    if len(ann_files) > 8:
        print(f"\n  ... and {len(ann_files) - 8} more text files")

    assert all_rows, "no parsable annotation rows found - inspect the ROOT layout"
    cls_rows = [(n, i) for n, i in all_rows if 1 <= len(i) <= 3]

    grade_idx, count_idx, infer = infer_columns(cls_rows)
    print("\n  --- INFERENCE REPORT (check before trusting the rest) ---")
    print(f"    line widths seen      : {infer['line_widths']}")
    for s in infer["column_stats"]:
        if s:
            print(f"    column {s['index']}: min={s['min']} max={s['max']} "
                  f"distinct={s['n_distinct']} mean={s['mean']:.2f}")
    print(f"    inferred GRADE column : {grade_idx}")
    print(f"    inferred COUNT column : {count_idx}")
    assert grade_idx is not None and count_idx is not None, (
        "could not infer columns - set grade_idx/count_idx by hand")

    records = {}
    for name, ints in cls_rows:
        if len(ints) > max(grade_idx, count_idx):
            records[Path(name).name] = {"grade": ints[grade_idx], "count": ints[count_idx]}

    counts = [r["count"] for r in records.values()]
    grades = [r["grade"] for r in records.values()]
    print(f"\n  images with grade+count : {len(records)}")
    print(f"  grade distribution      : {dict(sorted(Counter(grades).items()))}")
    print(f"  count range             : {min(counts)} .. {max(counts)}")
    print("  canonical reference     : 497 / 637 / 186 / 137 for grades 0-3")

    # ---- N1 --------------------------------------------------------------
    print("\n" + "=" * 74)
    print(f"N1  agreement rule(count) vs released grade, Hayashi cuts {HAYASHI_CUTS}")
    print("=" * 74)
    pred = [rule(c) for c in counts]
    n_exact = sum(1 for p, g in zip(pred, grades) if p == g)
    n1 = n_exact / len(grades)
    off_by = Counter(abs(p - g) for p, g in zip(pred, grades))
    print(f"  EXACT AGREEMENT = {n1:.4f}   ({n_exact}/{len(grades)})")
    print(f"  |delta| distribution = {dict(sorted(off_by.items()))}")

    cm = [[0] * N_GRADES for _ in range(N_GRADES)]
    for p, g in zip(pred, grades):
        if 0 <= g < N_GRADES and 0 <= p < N_GRADES:
            cm[g][p] += 1
    print("  rows = released grade, cols = rule(count):")
    for i, row in enumerate(cm):
        print(f"    g{i}: {row}")

    # ---- N2 --------------------------------------------------------------
    print("\n" + "=" * 74)
    print("N2  cut-points that best reproduce the released grades")
    print("=" * 74)
    fitted, fitted_acc = fit_cuts(counts, grades, max(counts))
    print(f"  FITTED CUTS     = {fitted}  (agreement {fitted_acc:.4f})")
    print(f"  PUBLISHED CUTS  = {HAYASHI_CUTS}  (agreement {n1:.4f})")
    print(f"  fitted / published ratio = {[round(f / h, 2) for f, h in zip(fitted, HAYASHI_CUTS)]}")
    print()
    fit_diag = describe_cuts(counts, grades, fitted, label="  ")
    print("\n  Ratios near 2.0 have TWO competing explanations:")
    print("    (a) full-face counting against half-face thresholds, or")
    print("    (b) annotation incompleteness - ACNE04-v2 has ~26.9 lesions/image")
    print("        against the original ~13.0, itself ~2.07x.")
    print("  Set V2_DETECTION_ROOT to separate them.")

    # ---- N3 --------------------------------------------------------------
    print("\n" + "=" * 74)
    print("N3  label count vs box count")
    print("=" * 74)
    boxes = count_boxes(root)
    print(f"  images with box annotations : {len(boxes)}")
    both = [(records[n]["count"], boxes[n]) for n in records if n in boxes]
    n3 = None
    if both:
        n = len(both)
        exact = sum(1 for a, b in both if a == b) / n
        mad = sum(abs(a - b) for a, b in both) / n
        ma, mb = sum(a for a, _ in both) / n, sum(b for _, b in both) / n
        va = sum((a - ma) ** 2 for a, _ in both) ** 0.5
        vb = sum((b - mb) ** 2 for _, b in both) ** 0.5
        cov = sum((a - ma) * (b - mb) for a, b in both)
        r = cov / (va * vb) if va and vb else float("nan")
        n3 = {"n": n, "exact": exact, "mad": mad, "pearson_r": r}
        print(f"  overlapping images          : {n}")
        print(f"  exact match label vs boxes  : {exact:.4f}")
        print(f"  mean |label - boxes|        : {mad:.2f}")
        print(f"  Pearson r                   : {r:.4f}")
        print("  If these disagree, state which quantity the detector route targets.")
    else:
        print("  no overlap between box annotations and classification rows.")

    # ---- N4 --------------------------------------------------------------
    print("\n" + "=" * 74)
    print("N4  subject groups and random-split leakage")
    print("=" * 74)
    groups = subject_groups(sorted(records))
    sizes = Counter(Counter(groups.values()).values())
    print(f"  inferred subject groups : {len(set(groups.values()))} for {len(groups)} images")
    print(f"  group-size histogram    : {dict(sorted(sizes.items()))}")
    rates = [leak_rate(groups, seed=s) for s in range(SEEDS)]
    mean_rate = sum(rates) / len(rates)
    sd = (sum((x - mean_rate) ** 2 for x in rates) / max(1, len(rates) - 1)) ** 0.5
    print(f"\n  LEAK RATE under random 80/20 split: {mean_rate:.4f} +/- {sd:.4f} over {SEEDS} seeds")
    print("  = fraction of test images with a same-subject image in train.")
    print("  Heuristic - over-merges. Upper bound on subjects, lower bound on leakage.")

    # ---- optional v2 -----------------------------------------------------
    if V2_DETECTION_ROOT:
        print("\n" + "=" * 74)
        print("V2  annotation density vs half-face convention")
        print("=" * 74)
        v2 = Path(V2_DETECTION_ROOT)
        assert v2.exists(), f"{v2} does not exist"
        boxes_v2 = count_boxes(v2)
        shared = [n for n in records if n in boxes_v2]
        print(f"  images shared with v2 : {len(shared)}")
        if shared:
            g_sh = [records[n]["grade"] for n in shared]
            c_v1 = [records[n]["count"] for n in shared]
            c_v2 = [boxes_v2[n] for n in shared]
            print(f"  mean lesions/image v1 : {sum(c_v1)/len(c_v1):.2f}")
            print(f"  mean lesions/image v2 : {sum(c_v2)/len(c_v2):.2f}")
            f1, a1 = fit_cuts(c_v1, g_sh, max(c_v1))
            f2, a2 = fit_cuts(c_v2, g_sh, max(c_v2))
            print(f"  fitted cuts v1 : {f1}  ({a1:.4f})")
            print(f"  fitted cuts v2 : {f2}  ({a2:.4f})")
            print(f"  v2 cuts / published : {[round(b / h, 2) for b, h in zip(f2, HAYASHI_CUTS)]}")
            print()
            describe_cuts(c_v1, g_sh, f1, label="  v1 ")
            print()
            describe_cuts(c_v2, g_sh, f2, label="  v2 ")
            print("\n  Near 1.0 means the 2x was annotation incompleteness.")
            print("  Near 2.0 means it is full-face counting.")

    # ---- verdict ---------------------------------------------------------
    print("\n" + "=" * 74)
    print("VERDICT ON THE COUNT ROUTE")
    print("=" * 74)
    if n1 > 0.97 or fitted_acc > 0.97:
        print("  ~100% reproducible: the grade IS the rule applied to the count.")
        print("  The count route is the label-generating function, not independent")
        print("  clinical evidence. Reframe: disagreement is detector-vs-classifier,")
        print("  and the BOUNDARY-MARGIN component carries the novelty.")
    elif fitted_acc >= 0.70:
        print("  Partially reproducible. Grades look clinician-assigned with the rule")
        print("  as guidance - compare against Hayashi's own 75.1% panel conformity.")
        print("  'Count-grounded reliability' is defensible broadly as written.")
    else:
        print("  Poorly reproducible. The counts are probably a different quantity")
        print("  than the grading rule uses, or the grades are noisy. Characterise")
        print("  the residual before building anything on top of it.")
    print("=" * 74)

    # ---- save ------------------------------------------------------------
    summary = {
        "inventory": {"image_files": len(images), "unique_filenames": len(unique),
                      "published_canonical": CANONICAL, "reported_zip_contents": REPORTED_ZIP,
                      "images_per_directory": dict(by_dir)},
        "n1_published_rule_agreement": n1,
        "n1_abs_delta_hist": {str(k): v for k, v in sorted(off_by.items())},
        "n1_confusion_released_vs_rule": cm,
        "n2_fitted_cuts": list(fitted),
        "n2_fitted_agreement": fitted_acc,
        "n2_published_cuts": list(HAYASHI_CUTS),
        "n2_fit_diagnostics": fit_diag,
        "n3_label_vs_boxes": n3,
        "n4_subject_groups": len(set(groups.values())),
        "n4_leak_rate_mean": mean_rate,
        "n4_leak_rate_sd": sd,
        "inference_report": infer,
        "grade_distribution": {str(k): v for k, v in sorted(Counter(grades).items())},
    }
    (out / "week1_audit.json").write_text(json.dumps(summary, indent=2, default=str))

    with (out / "per_image.csv").open("w") as fh:
        fh.write("image,released_grade,label_count,rule_grade_published,"
                 "rule_grade_fitted,box_count,subject_group\n")
        for n in sorted(records):
            rec = records[n]
            fh.write(f"{n},{rec['grade']},{rec['count']},{rule(rec['count'])},"
                     f"{rule(rec['count'], fitted)},{boxes.get(n, '')},{groups.get(n, '')}\n")

    print(f"\nwrote {out / 'week1_audit.json'}")
    print(f"wrote {out / 'per_image.csv'}")
    return 0

_ = main()

SETUP
  2934 extracted files already present - skipping extraction

  TREE (depth 2, recursive file counts)
    Classification/   (1467 files)
        Classification/Classification/   (1467 files)
    Detection/   (1467 files)
        Detection/Detection/   (1467 files)

  FILE TYPES
    .xml       1457
    .jpg       1457
    .txt       20

IMAGE INVENTORY
  image files on disk    : 1457
  UNIQUE filenames       : 1457   <- the real inventory
  published canonical    : 1457
  reported zip contents  : 1513

  images per directory:
      1457  Classification/Classification/JPEGImages/

  Unique count matches the canonical 1457.

ANNOTATION FILES

  Classification/Classification/NNEW_test_0.txt   (292 lines)
      levle0_451.jpg  0  2
      levle0_498.jpg  0  1
      levle0_485.jpg  0  3
      levle0_218.jpg  0  2
      levle1_344.jpg  0  4

  Classification/Classification/NNEW_test_1.txt   (292 lines)
      levle0_60.jpg  0  1
      levle0_483.jpg  0  2
      levle0_110.jpg  0  2
      

# Boundary-margin feasibility gate

ACNE04's grade is exactly `rule(count)` under Hayashi cut-points (5, 20, 50) — confirmed at **1457/1457**. So the label is a *threshold of a count*, and label ambiguity is structurally concentrated near the cut-points: an image at count 21 is one lesion from a grade boundary, one at count 35 is not.

That is the proposed reliability signal. This notebook asks whether it has room to work, before anything is built on top of it.

| | |
|---|---|
| **Q1** | how many images sit within *k* lesions of a cut-point |
| **Q2** | grade-flip rate if the detector miscounts by ±k |
| **Q3** | which grades carry the boundary risk |
| **Q4** | ceiling AUROC for predicting a wrong grade from margin alone |

Reads the `per_image.csv` from the Week-1 audit. Standard library only, runs in seconds.

### Pre-registered interpretation

Decided **before** looking at the output, so the decision is not made after the fact:

| Q2 at ±3 | Meaning |
|---|---|
| above ~25% | the signal has real room — build the method |
| 10–25% | workable but the effect will be modest; measure the detector's real error distribution before promising an effect size |
| below ~10% | boundary margin cannot carry a paper on this dataset — the audit becomes the contribution |


## 1. Config

In [4]:
from __future__ import annotations

import csv
import random
from collections import Counter
from pathlib import Path

# --- edit ------------------------------------------------------------------
PER_IMAGE_CSV = "audit_out/per_image.csv"
CUTS = (5, 20, 50)          # confirmed exactly by N1
TRIALS = 2000               # Monte-Carlo trials for Q2 and Q4
# ---------------------------------------------------------------------------

## 2. Helpers

In [5]:
N_GRADES = len(CUTS) + 1


def rule(count, cuts=CUTS):
    g = 0
    for c in cuts:
        if count > c:
            g += 1
        else:
            break
    return g


def margin(count, cuts=CUTS):
    """
    Lesions away from flipping grade: how far the count can move before
    rule(count) changes. Grade boundaries sit between c and c+1.
    """
    best = None
    for c in cuts:
        # distance to the nearest side of this boundary
        d = abs(count - c) if count <= c else abs(count - (c + 1))
        best = d if best is None else min(best, d)
    return best

## 3. Run

In [6]:
def main():
    path = Path(PER_IMAGE_CSV)
    if not path.exists():
        print(f"ERROR: {path} not found. Run the Week-1 audit first.")
        return 2

    counts, grades = [], []
    with path.open() as fh:
        for row in csv.DictReader(fh):
            try:
                counts.append(int(row["label_count"]))
                grades.append(int(row["released_grade"]))
            except (KeyError, ValueError):
                continue
    n = len(counts)
    assert n, "no usable rows in per_image.csv"

    print("=" * 74)
    print(f"BOUNDARY-MARGIN GATE   n={n}   cuts={CUTS}")
    print("=" * 74)

    mismatch = sum(1 for c, g in zip(counts, grades) if rule(c) != g)
    print(f"  sanity: rule(count) != released grade on {mismatch} images")
    if mismatch:
        print("  WARNING: cut-points do not reproduce the grades. Fix CUTS first.")

    margins = [margin(c) for c in counts]

    # ---- Q1 --------------------------------------------------------------
    print("\n" + "-" * 74)
    print("Q1  how many images sit within k lesions of a cut-point")
    print("-" * 74)
    print(f"  {'k':>3}  {'within k':>9}  {'share':>7}")
    q1 = {}
    for k in (0, 1, 2, 3, 5, 10):
        c = sum(1 for m in margins if m <= k)
        q1[k] = c / n
        print(f"  {k:>3}  {c:>9}  {c/n:>6.1%}")

    hist = Counter(margins)
    print("\n  margin histogram (0 = adjacent to a boundary):")
    for m in sorted(hist)[:15]:
        bar = "#" * max(1, round(60 * hist[m] / max(hist.values())))
        print(f"    {m:>3}  {hist[m]:>5}  {bar}")
    if max(hist) > 14:
        tail = sum(v for k, v in hist.items() if k > 14)
        print(f"    >14  {tail:>5}")

    # ---- Q2 --------------------------------------------------------------
    print("\n" + "-" * 74)
    print("Q2  grade-flip rate if the detector miscounts")
    print("-" * 74)
    print("  Deterministic worst case: flips if margin <= k.")
    print("  Monte-Carlo: count error drawn uniformly from [-k, +k].\n")
    print(f"  {'+/-k':>5}  {'worst case':>11}  {'uniform error':>14}")
    rng = random.Random(0)
    q2 = {}
    for k in (1, 2, 3, 5):
        worst = sum(1 for m in margins if m <= k) / n
        flips = 0
        for _ in range(TRIALS):
            i = rng.randrange(n)
            noisy = max(0, counts[i] + rng.randint(-k, k))
            if rule(noisy) != grades[i]:
                flips += 1
        mc = flips / TRIALS
        q2[k] = {"worst_case": worst, "uniform": mc}
        print(f"  {k:>5}  {worst:>10.1%}  {mc:>13.1%}")

    # ---- Q3 --------------------------------------------------------------
    print("\n" + "-" * 74)
    print("Q3  which grades carry the boundary risk")
    print("-" * 74)
    print(f"  {'grade':>5}  {'n':>5}  {'median margin':>13}  {'share margin<=2':>16}")
    q3 = {}
    for g in range(N_GRADES):
        ms = sorted(m for m, gg in zip(margins, grades) if gg == g)
        if not ms:
            continue
        med = ms[len(ms) // 2]
        share = sum(1 for m in ms if m <= 2) / len(ms)
        q3[g] = {"n": len(ms), "median_margin": med, "share_le_2": share}
        print(f"  {g:>5}  {len(ms):>5}  {med:>13}  {share:>15.1%}")

    # ---- Q4 --------------------------------------------------------------
    print("\n" + "-" * 74)
    print("Q4  ceiling on error detection from margin alone")
    print("-" * 74)
    print("  Simulates a detector with uniform count error, then asks how well")
    print("  a LOW MARGIN predicts that the grade came out wrong (AUROC).")
    print("  This is an optimistic ceiling: the real detector's error is")
    print("  count-dependent, not uniform, and must be measured, not assumed.\n")
    print(f"  {'+/-k':>5}  {'error rate':>11}  {'AUROC':>7}")
    q4 = {}
    for k in (2, 3, 5):
        errs, scores = [], []
        rng = random.Random(1)
        for i in range(n):
            noisy = max(0, counts[i] + rng.randint(-k, k))
            errs.append(1 if rule(noisy) != grades[i] else 0)
            scores.append(-margins[i])          # low margin = high risk
        pos = [s for s, e in zip(scores, errs) if e]
        neg = [s for s, e in zip(scores, errs) if not e]
        if not pos or not neg:
            print(f"  {k:>5}  {'n/a':>11}  {'n/a':>7}")
            continue
        # rank-based AUROC with ties at 0.5
        wins = sum((p > q) + 0.5 * (p == q) for p in pos for q in neg)
        auroc = wins / (len(pos) * len(neg))
        q4[k] = {"error_rate": len(pos) / n, "auroc": auroc}
        print(f"  {k:>5}  {len(pos)/n:>10.1%}  {auroc:>7.3f}")

    # ---- verdict ---------------------------------------------------------
    print("\n" + "=" * 74)
    print("VERDICT")
    print("=" * 74)
    key = q2[3]["uniform"]
    print(f"  Grade-flip rate at +/-3 lesions: {key:.1%}")
    if key > 0.25:
        print("  ROOM TO WORK. A meaningful share of images are boundary-fragile,")
        print("  so margin can separate them. Build the method.")
    elif key > 0.10:
        print("  WORKABLE BUT MODEST. Enough fragile images to measure, not enough")
        print("  for a large effect. Measure the detector's real count-error")
        print("  distribution before promising an effect size.")
    else:
        print("  TOO FEW FRAGILE IMAGES. Boundary margin cannot carry a method on")
        print("  this dataset: most images are far from a cut-point, so the signal")
        print("  has nothing to flag. Make the dataset audit the contribution.")
    print()
    print(f"  Images within 2 lesions of a boundary: {q1[2]:.1%}")
    print(f"  Ceiling AUROC from margin alone (+/-3): "
          f"{q4.get(3, {}).get('auroc', float('nan')):.3f}")
    print("=" * 74)
    print("\n  Caveat: uniform count error is a placeholder. The real detector")
    print("  under-counts on dense faces, so error grows with count and the true")
    print("  flip rate will differ. Re-run with the measured error distribution")
    print("  once the detector exists.")
    return 0

_ = main()

BOUNDARY-MARGIN GATE   n=1457   cuts=(5, 20, 50)
  sanity: rule(count) != released grade on 0 images

--------------------------------------------------------------------------
Q1  how many images sit within k lesions of a cut-point
--------------------------------------------------------------------------
    k   within k    share
    0        231   15.9%
    1        417   28.6%
    2        632   43.4%
    3        899   61.7%
    5       1253   86.0%
   10       1397   95.9%

  margin histogram (0 = adjacent to a boundary):
      0    231  ###################################################
      1    186  #########################################
      2    215  ###############################################
      3    267  ##########################################################
      4    274  ############################################################
      5     80  ##################
      6     57  ############
      7     45  ##########
      8     15  ###
      9     1